# 04 - Plata: compuertas de calidad, deduplicación y cuarentena

**Tarea `plata_calidad`.** Stream desde `bronze.ais_raw` con `foreachBatch`:

1. Tipificación tolerante (`try_cast`) y 12 expectativas (7 CUARENTENA, 5 ADVERTENCIA) de `oceanwatch/quality.py`.
2. Deduplicación por (MMSI, BaseDateTime) dentro del lote y frente a lotes previos (D01 exacto / D02 conflictivo).
3. Escrituras idempotentes (`txnAppId`, `txnVersion=batchId`) a `silver.ais_posiciones`, `silver.ais_cuarentena` y `silver.metricas_calidad`.

Los registros inválidos se enrutan a cuarentena con su causa **sin detener el stream**.

In [0]:
import os
import sys

# El código del pipeline vive en ../src (misma librería probada en local; ver README).
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from oceanwatch.lakehouse import get_lakehouse
from oceanwatch.notebook_params import fechas, widget

CATALOG = widget(dbutils, "catalogo", "oceanwatch_g06")
lh = get_lakehouse(spark, catalog=CATALOG)

In [0]:
import json

from pyspark.sql import functions as F

from oceanwatch.streaming import run_silver

print(json.dumps(run_silver(lh), indent=2))
display(lh.read("silver", "metricas_calidad").orderBy(F.desc("batch_id"), "regla").limit(40))

{
  "query": "silver_ais_posiciones",
  "lotes_con_datos": 0,
  "filas_entrada": 0,
  "ids_lote": []
}


batch_id,processed_at,ingestion_date,regla,severidad,descripcion,registros_evaluados,registros_fallidos,porcentaje_fallidos
2,2026-11-19T17:37:30.610Z,2023-06-10,D01_DUPLICADO_EXACTO,DEDUPLICACION,"Segunda llegada de (MMSI, BaseDateTime) con contenido idéntico.",7762336,177,0.0023
2,2026-11-19T17:37:30.610Z,2023-06-10,D02_DUPLICADO_CONFLICTIVO,DEDUPLICACION,"Segunda llegada de (MMSI, BaseDateTime) con contenido distinto; se conserva la primera.",7762336,71,9.0E-4
2,2026-11-19T17:37:30.610Z,2023-06-10,E01_MMSI_FORMATO,CUARENTENA,MMSI numérico de exactamente 9 dígitos (D-07).,7794117,6593,0.0846
2,2026-11-19T17:37:30.610Z,2023-06-10,E02_TIMESTAMP_VALIDO,CUARENTENA,BaseDateTime presente y con formato yyyy-MM-ddTHH:mm:ss.,7794117,0,0.0
2,2026-11-19T17:37:30.610Z,2023-06-10,E03_TIMESTAMP_EN_VENTANA,CUARENTENA,La fecha UTC de la posición coincide con el día del archivo del feed.,7794117,0,0.0
2,2026-11-19T17:37:30.610Z,2023-06-10,E04_COORDENADAS_PRESENTES,CUARENTENA,LAT y LON presentes y numéricas.,7794117,0,0.0
2,2026-11-19T17:37:30.610Z,2023-06-10,E05_COORDENADAS_EN_RANGO,CUARENTENA,"LAT en [-90, 90], LON en [-180, 180] y distinta de (0, 0) (isla nula GPS).",7794117,0,0.0
2,2026-11-19T17:37:30.610Z,2023-06-10,E06_COORDENADAS_EN_AGUA,CUARENTENA,La posición no cae en una celda H3 r5 de tierra firme (máscara Natural Earth erosionada).,7794117,25117,0.3223
2,2026-11-19T17:37:30.610Z,2023-06-10,E07_SOG_PLAUSIBLE,CUARENTENA,"SOG presente y en [0, 60] nudos, o el sentinela 102.3 (no disponible).",7794117,77,0.001
2,2026-11-19T17:37:30.610Z,2023-06-10,E08_COG_RANGO,ADVERTENCIA,"COG en [0, 360]; 360 es el sentinela no disponible (se anula sin contarse como error).",7794117,0,0.0
